# Install & Import Libraries

In [1]:
!pip install -q transformers datasets
!pip install -q --upgrade transformers

import json
import pandas as pd
from datasets import Dataset
from transformers import T5Tokenizer, T5ForConditionalGeneration, Trainer, TrainingArguments

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 78.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 660.6/660.6 kB 38.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 100.6 MB/s eta 0:00:00


# Load SQuAD Dataset

In [2]:
DATA_PATH = "/kaggle/input/datasets/organizations/stanfordu/stanford-question-answering-dataset"

# Load JSON

In [3]:
def load_squad(file_path):
    with open(file_path, 'r') as f:
        data = json.load(f)

    contexts, questions, answers = [], [], []

    for item in data['data']:
        for paragraph in item['paragraphs']:
            context = paragraph['context']
            for qa in paragraph['qas']:
                question = qa['question']
                answer = qa['answers'][0]['text'] if qa['answers'] else ""

                contexts.append(context)
                questions.append(question)
                answers.append(answer)

    return pd.DataFrame({
        "context": contexts,
        "question": questions,
        "answer": answers
    })

In [4]:
train_df = load_squad(f"{DATA_PATH}/train-v1.1.json")
val_df   = load_squad(f"{DATA_PATH}/dev-v1.1.json")

print(train_df.head())

                                             context  \
0  Architecturally, the school has a Catholic cha...   
1  Architecturally, the school has a Catholic cha...   
2  Architecturally, the school has a Catholic cha...   
3  Architecturally, the school has a Catholic cha...   
4  Architecturally, the school has a Catholic cha...   

                                            question  \
0  To whom did the Virgin Mary allegedly appear i...   
1  What is in front of the Notre Dame Main Building?   
2  The Basilica of the Sacred heart at Notre Dame...   
3                  What is the Grotto at Notre Dame?   
4  What sits on top of the Main Building at Notre...   

                                    answer  
0               Saint Bernadette Soubirous  
1                a copper statue of Christ  
2                        the Main Building  
3  a Marian place of prayer and reflection  
4       a golden statue of the Virgin Mary  


In [5]:
print("Shape:")
print(train_df.shape)

Shape:
(87599, 3)


In [6]:
print("Info:")
train_df.info()

Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 87599 entries, 0 to 87598
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   context   87599 non-null  object
 1   question  87599 non-null  object
 2   answer    87599 non-null  object
dtypes: object(3)
memory usage: 2.0+ MB


In [7]:
print("Describe:")
print(train_df.describe())

Describe:
                                                  context  \
count                                               87599   
unique                                              18891   
top     In 1853, Victoria gave birth to her eighth chi...   
freq                                                   25   

                                                 question answer  
count                                               87599  87599  
unique                                              87355  65134  
top     Which Caribbean nation is in the top quartile ...  three  
freq                                                    6    233  


# Convert to HuggingFace Dataset

In [8]:
train_dataset = Dataset.from_pandas(train_df)
val_dataset   = Dataset.from_pandas(val_df)

# Load T5 Model & Tokenizer

In [9]:
model_name = "t5-small"

tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

# Preprocessing Function

In [10]:
def preprocess(example):
    input_text = "question: " + example["question"] + " context: " + example["context"]
    target_text = example["answer"]

    inputs = tokenizer(
        input_text,
        max_length=512,
        truncation=True,
        padding="max_length"
    )

    targets = tokenizer(
        target_text,
        max_length=64,
        truncation=True,
        padding="max_length"
    )

    inputs["labels"] = targets["input_ids"]
    return inputs

#  Apply preprocessing

In [11]:
train_dataset = train_dataset.map(preprocess, batched=False)
val_dataset   = val_dataset.map(preprocess, batched=False)

train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
val_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

Map:   0%|          | 0/87599 [00:00<?, ? examples/s]

Map:   0%|          | 0/10570 [00:00<?, ? examples/s]

# Training Arguments

In [12]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=2,
    eval_strategy="epoch",   # ✅ FIX (NOT evaluation_strategy)
    save_strategy="epoch",
    logging_dir="./logs",
    logging_steps=100,
    save_total_limit=2,
    fp16=True
)


[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


# Trainer & Train Model

In [13]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss
1,0.072735,0.071220
2,0.063138,0.070200


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=21900, training_loss=0.11334506770791529, metrics={'train_runtime': 5247.482, 'train_samples_per_second': 33.387, 'train_steps_per_second': 4.173, 'total_flos': 2.3711612934291456e+16, 'train_loss': 0.11334506770791529, 'epoch': 2.0})

# Save Model & Tokenizer

In [14]:
model.save_pretrained("/kaggle/working/t5_qa_model")
tokenizer.save_pretrained("/kaggle/working/t5_qa_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/kaggle/working/t5_qa_model/tokenizer_config.json',
 '/kaggle/working/t5_qa_model/tokenizer.json')

# Load Model and Tokenizer

In [15]:
model = T5ForConditionalGeneration.from_pretrained("/kaggle/working/t5_qa_model")
tokenizer = T5Tokenizer.from_pretrained("/kaggle/working/t5_qa_model")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

# Prediction Function

In [16]:
def predict(question, context):
    input_text = "question: " + question + " context: " + context

    input_ids = tokenizer.encode(input_text, return_tensors="pt", truncation=True)

    outputs = model.generate(input_ids, max_length=64)
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return answer

# Test Prediction

In [17]:
sample = val_df.iloc[5]

print("Question:", sample["question"])
print("Actual Answer:", sample["answer"])

pred = predict(sample["question"], sample["context"])
print("Predicted Answer:", pred)

Question: What was the theme of Super Bowl 50?
Actual Answer: "golden anniversary"
Predicted Answer: gold
